# Solution E1 · Optimizing and evaluating on the same data

**Type**: Find the bug

## Objective

Find out why a random forest tuned with `GridSearchCV` "gets" an AUC of 0.95 on the test set on Titanic, fix the
scheme (split, search on the training set only, test set touched once) and measure how much the number was inflated.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = ds.titanic()
X, y = df.drop(columns="survived"), df["survived"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
NUM = ["age", "siblings_spouses", "parents_children", "fare"]
CAT = ["sex", "port", "deck", "title"]
cv = StratifiedKFold(5, shuffle=True, random_state=0)

prep = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), NUM),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), CAT),
    ("pclass", "passthrough", ["pclass"]),
], sparse_threshold=0)
pipe = make_pipeline(prep, RandomForestClassifier(random_state=0))
grid = {"randomforestclassifier__max_depth": [4, 8, None], "randomforestclassifier__min_samples_leaf": [1, 3]}

## Commented solution

In [ ]:
bug_search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X, y)             # X, y: ALL the data
auc_bug = roc_auc_score(y_te, bug_search.predict_proba(X_te)[:, 1])
print(f"test set AUC: {auc_bug:.3f}   (best_score_ of the search: {bug_search.best_score_:.3f})")

**Where the bug is.** `X, y` also contain the rows of `X_te`: the search trained (with `refit=True` the final model is trained
on everything) on exactly the passengers on which it is then "evaluated". A random forest with no depth limit memorized them: AUC close to 1.
It is the evaluation of a model on its own training data.

In [ ]:
test_share_in_training = float(X_te.index.isin(X.index).mean())
ok_search = GridSearchCV(pipe, grid, cv=cv, scoring="roc_auc").fit(X_tr, y_tr)         # the search sees ONLY the training set
auc_ok = roc_auc_score(y_te, ok_search.predict_proba(X_te)[:, 1])                       # the test set, just once
best_ok = ok_search.best_score_
inflation = auc_bug - auc_ok
print(f"rows of the test set present in the colleague's search: {test_share_in_training:.0%}")
print(f"correct test AUC: {auc_ok:.3f} | best_score_ (cv on the training set): {best_ok:.3f} | inflation: {inflation:.3f}")

## Check

In [ ]:
with verify():
    check(auc_bug > 0.93, "the colleague's number is too good for the Titanic: it is leakage", "auc_bug should exceed 0.93; don't modify the colleague's code")
    check(test_share_in_training == 1.0, "all the rows of the test set were in the search", "test_share_in_training = X_te.index.isin(X.index).mean()")
    check(auc_ok < 0.92, "with the search on the training set only the AUC is realistic (below 0.92)", "the search must see only X_tr, y_tr")
    check(close(inflation, auc_bug - auc_ok) and inflation > 0.05, "the colleague overestimated the performance by more than 0.05 of AUC",
          "inflation = auc_bug - auc_ok")
    check(abs(best_ok - auc_ok) < 0.05, "best_score_ (cross-validation) and the correct test AUC are close", "best_ok = best_score_ of the search on the training set alone")
done()

## What you learned

- With `refit=True` the search retrains the best one on **everything** it has seen. If it saw the test set, the model learned it:
  an unregularized random forest memorizes it (AUC close to 1).
- The correct scheme has three sets with distinct roles: **training** (the search does its cross-validation inside), **test** (a single
  final evaluation), and the inner folds (disposable). The test set must not influence anything, not even the choice of hyperparameters.
- A symptom of this bug is a score that is "too good": AUC 0.95 on the test set when the cross-validation says 0.87 (and the honest AUC is 0.88) is a warning sign. Comparing
  the test score with `best_score_` and with the baseline is the cheapest check.

## Common pitfall

Redoing the search on all the data "after choosing" to produce the final model and then reporting the earlier test score. It is fine only
if the reported score is the one obtained **before** retraining on everything: afterward, the test set is no longer independent.

## Interview question

<details><summary>After evaluating on the test set, can I retrain the model on training + test before putting it in production?</summary>

Yes, but the score to report is the test set score obtained **before** retraining: it is the honest estimate of the chosen
model. The final model trained on more data will probably be a bit better, but I no longer have an independent set to
measure it. If I have to change something after seeing the test set, the test set is "burned" and I need a new holdout set.

</details>